# Alfa-Code Tiny-50M — Colab Free T4
Runtime → Change runtime type → T4 GPU. Then Run all.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q torch transformers datasets tokenizers accelerate huggingface_hub pyyaml bitsandbytes

In [ ]:
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU not visible to torch. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart session and Run all.')


In [ ]:
from huggingface_hub import login
from google.colab import userdata
try:
    tok = userdata.get('HF_TOKEN')
except Exception:
    tok = None
if tok:
    login(token=tok)
    print('logged in')
else:
    print('No HF_TOKEN secret — Hub upload step will be skipped. Add it via the key icon if needed.')


In [ ]:
# Code comes from public GitHub (no token needed) — avoids 401 from private HF repo
%cd /content
!rm -rf /content/alfa
!git clone https://github.com/mrsandipmandal/alfa-code.git /content/alfa
%cd /content/alfa
!ls scripts/ configs/

In [ ]:
# Pick ONE dataset. Start small (500 rows) to verify, then scale up.
# Options: code_long | image_to_code | video_to_code | multimodal_mix | code_small_test
!python scripts/download_data.py --dataset multimodal_mix --max-rows 1500 --overwrite
!python scripts/train_tokenizer.py
!ls -lh tokenizers/ data/processed/

In [ ]:
!python scripts/train_hf.py --data data/processed/train.jsonl --out /content/outputs/tiny-50M --epochs 1 --config configs/tiny-50M.yaml --max-seq-len 2048 --batch-size 2
!ls -lh /content/outputs/tiny-50M

In [ ]:
# Push back to Hub (uploads whichever model was trained: tiny-50M and/or base-1B).
# Needs HF_TOKEN in Colab Secrets: key icon on the left.
import os
from pathlib import Path
tok = None
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
except Exception:
    tok = os.getenv('HF_TOKEN')
if not tok:
    print('SKIP: no HF_TOKEN secret set.')
else:
    from huggingface_hub import HfApi
    api = HfApi(token=tok)
    for name in ['tiny-50M', 'base-1B']:
        local = Path(f'/content/outputs/{name}')
        if (local / 'config.json').exists():
            api.upload_folder(folder_path=str(local), repo_id='mrsandip/Alfa-Code', path_in_repo=f'outputs/{name}')
            print('uploaded', name)
        else:
            print('skip (not trained yet):', local)


In [ ]:
# OPTIONAL scale-up: more rows + chat tuning + 1B model (needs T4 16GB, ~2-4h).
# Run this INSTEAD of the default data/train cells above.
!python scripts/download_data.py --dataset multimodal_mix --max-rows 5000 --overwrite
!python scripts/download_data.py --dataset chat_qa --max-rows 2000
!python scripts/train_tokenizer.py
!python scripts/train_hf.py --data data/processed/train.jsonl --out /content/outputs/base-1B --epochs 3 --config configs/base-1B.yaml --max-seq-len 1024 --batch-size 1  # 1024 ctx fits 1B on T4; 2048 OOMs
!ls -lh /content/outputs/base-1B


In [ ]:
# Launch UI: pulls latest code, then serves the newest trained model below (else mock).
# Open the gradio.live link that appears below.
%cd /content/alfa
!git pull origin main
import os
from pathlib import Path
for _cand in ['/content/outputs/base-1B', '/content/outputs/tiny-50M']:
    if (Path(_cand) / 'config.json').exists():
        os.environ['ALFA_MODEL'] = _cand
        print('using model:', _cand)
        break
import importlib, app.model_backend as _mb, app.ui as _ui
importlib.reload(_mb); importlib.reload(_ui)
_ui.launch_share()
